# Rossmann Demand Forecasting: Full Pipeline

This notebook runs the whole project from the raw CSVs, top to bottom.

1. Setup
2. Load and basic data checks
3. Store coverage, gaps and random test stores
4. Clean and save the open-day table
5. Daily panel and features
6. Baselines on the latest holdout: same-weekday mean and ARIMA (Prophet is skipped, see Setup)
7. Gradient boosting on three rolling windows, with a same-weekday check and a window-size diagnostic
8. Accuracy across windows and head-to-head on the latest window
9. Promotion ablation
10. Inventory cost

Leakage rule: every lag is at least 28 days old, so a forecast only uses sales known on the last training day. Promotions, holidays and day of week are known in advance, so they are allowed. Customer counts are not, so they are excluded.

Closed days are kept as zero sales on the full calendar, so weekly lags are correct. Scoring uses open days with positive sales only.

## 1. Setup

In [1]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")

# Prophet is skipped: its bundled TBB library conflicts with a system DLL on this
# Windows machine. Set to True only after a Prophet fit works in this environment.
HAS_PROPHET = False

ROOT = Path(r"C:\Users\reinh\Downloads\Projects\Rossmann")
RAW_TRAIN = ROOT / "Rossmann" / "train.csv"
RAW_STORE = ROOT / "Rossmann" / "store.csv"
PROC = ROOT / "processed"
PROC.mkdir(parents=True, exist_ok=True)

HORIZON = 28
WINDOW_OFFSETS = [0, 28, 56]      # 0 = latest window; 28 and 56 days earlier
MARGINS = [0.15, 0.25, 0.35]      # gross margin as a share of sales value (assumption)

assert RAW_TRAIN.exists(), f"Not found: {RAW_TRAIN}"
assert RAW_STORE.exists(), f"Not found: {RAW_STORE}"

## 2. Load and basic data checks

In [2]:
# StateHoliday mixes 0 and "0" with letters, so read it as text.
train = pd.read_csv(RAW_TRAIN, dtype={"StateHoliday": str}, parse_dates=["Date"])
store = pd.read_csv(RAW_STORE)
calendar = pd.date_range(train["Date"].min(), train["Date"].max(), freq="D")

EXPECTED_TRAIN = ["Store", "DayOfWeek", "Date", "Sales", "Customers", "Open", "Promo", "StateHoliday", "SchoolHoliday"]
assert set(EXPECTED_TRAIN) <= set(train.columns), set(EXPECTED_TRAIN) - set(train.columns)

print("train:", train.shape, "| store:", store.shape)
print("Date range:", calendar[0].date(), "to", calendar[-1].date(), "| calendar days:", len(calendar))
print("Missing values:", train.isna().sum().sum())
print("Duplicate Store+Date rows:", train.duplicated(["Store", "Date"]).sum())
print("Negative sales:", (train["Sales"] < 0).sum())

closed = train["Open"] == 0
print("Closed days:", closed.sum(), f"({closed.mean():.1%} of rows)")
print("Closed days with sales > 0:", (closed & (train["Sales"] > 0)).sum())
print("Open days with zero sales:", ((~closed) & (train["Sales"] == 0)).sum())
print("\nStateHoliday values:\n", train["StateHoliday"].value_counts())

train: (1017209, 9) | store: (1115, 10)
Date range: 2013-01-01 to 2015-07-31 | calendar days: 942
Missing values: 0
Duplicate Store+Date rows: 0
Negative sales: 0
Closed days: 172817 (17.0% of rows)
Closed days with sales > 0: 0
Open days with zero sales: 54

StateHoliday values:
 StateHoliday
0    986159
a     20260
b      6690
c      4100
Name: count, dtype: int64


## 3. Store coverage, gaps and random test stores

Coverage is the share of calendar days with a row for the store. Closed days have a row, so they count as covered. A missing row means the store is absent from the file. Stores with full coverage and no gap of 14 or more days are eligible. Test stores are a seeded random sample from the eligible stores, so the choice is repeatable.

In [3]:
def longest_missing_run(dates):
    missing = calendar.difference(pd.DatetimeIndex(dates))
    if len(missing) == 0:
        return 0
    run = (pd.Series(missing).diff().dt.days != 1).cumsum()
    return int(run.value_counts().max())

coverage = train.groupby("Store")["Date"].nunique() / len(calendar)
longest_gap = train.groupby("Store")["Date"].apply(longest_missing_run)
stores = pd.DataFrame({"coverage": coverage, "longest_gap_days": longest_gap})
eligible = stores[(stores["coverage"] >= 0.95) & (stores["longest_gap_days"] < 14)]

print("Stores with a gap of 14+ days:", (stores["longest_gap_days"] >= 14).sum())
print("Eligible stores:", len(eligible), "of", len(stores))

rng = np.random.default_rng(42)
TEST_STORES = sorted(rng.choice(eligible.index.values, size=5, replace=False).tolist())
assert TEST_STORES == [98, 480, 486, 735, 865], f"Test stores changed: {TEST_STORES}"
print("Test stores:", TEST_STORES)

sunday_open = (
    train[(train["DayOfWeek"] == 7) & train["Store"].isin(TEST_STORES)]
    .groupby("Store")["Open"].mean().round(2)
)
print("Share of Sundays open:\n", sunday_open)

Stores with a gap of 14+ days: 180
Eligible stores: 935 of 1115
Test stores: [98, 480, 486, 735, 865]
Share of Sundays open:
 Store
98     0.0
480    0.0
486    0.0
735    0.0
865    0.0
Name: Open, dtype: float64


## 4. Clean and save the open-day table

This keeps open days with positive sales and adds store details. It's saved for reference. The forecasting below builds its own full-calendar panel, so it does not read this file.

In [4]:
df = train.merge(store, on="Store", how="left")
assert len(df) == len(train), "Merge changed the row count"

clean = df[(df["Open"] == 1) & (df["Sales"] > 0)].copy()
clean = clean.sort_values(["Store", "Date"]).reset_index(drop=True)

print("Rows before:", f"{len(df):,}", "| after:", f"{len(clean):,}")
print("Share dropped:", f"{1 - len(clean) / len(df):.1%}")
print(clean["Sales"].describe().round(1))

CLEAN_OUT = PROC / "rossmann_clean.csv"
clean.to_csv(CLEAN_OUT, index=False)
print("Saved:", CLEAN_OUT)

Rows before: 1,017,209 | after: 844,338
Share dropped: 17.0%
count    844338.0
mean       6956.0
std        3103.8
min          46.0
25%        4859.0
50%        6369.0
75%        8360.0
max       41551.0
Name: Sales, dtype: float64
Saved: C:\Users\reinh\Downloads\Projects\Rossmann\processed\rossmann_clean.csv


## 5. Daily panel and features

Every eligible store gets a row for every calendar day, so lags count calendar days. Missing rows stay NaN. All lags are 28 days or more.

In [5]:
STORES = eligible.index.values
panel = (
    train.set_index(["Store", "Date"])
    .reindex(pd.MultiIndex.from_product([STORES, calendar], names=["Store", "Date"]))
    .reset_index()
)
panel["DayOfWeek"] = panel["Date"].dt.dayofweek + 1  # 1 = Monday, 7 = Sunday
panel = panel.merge(store, on="Store", how="left")
panel = panel.sort_values(["Store", "Date"]).reset_index(drop=True)

g = panel.groupby("Store")["Sales"]
for k in [28, 35, 42, 49]:
    panel[f"lag_{k}"] = g.shift(k)
panel["same_dow_mean_4w"] = panel[["lag_28", "lag_35", "lag_42", "lag_49"]].mean(axis=1)
panel["roll_mean_28"] = g.transform(
    lambda s: s.shift(HORIZON).rolling(HORIZON, min_periods=14).mean()
)

panel["month"] = panel["Date"].dt.month
panel["day_of_month"] = panel["Date"].dt.day
panel["week_of_year"] = panel["Date"].dt.isocalendar().week.astype(int)
panel["StateHoliday"] = panel["StateHoliday"].fillna("0").map({"0": 0, "a": 1, "b": 2, "c": 3})
panel["StoreType"] = panel["StoreType"].astype("category").cat.codes
panel["Assortment"] = panel["Assortment"].astype("category").cat.codes
panel["CompetitorMissing"] = panel["CompetitionDistance"].isna().astype(int)
panel["CompetitionDistance"] = panel["CompetitionDistance"].fillna(100_000)

FEATURES = [
    "DayOfWeek", "Promo", "StateHoliday", "SchoolHoliday",
    "month", "day_of_month", "week_of_year",
    "StoreType", "Assortment", "CompetitionDistance", "CompetitorMissing", "Promo2",
    "lag_28", "lag_35", "lag_42", "lag_49", "same_dow_mean_4w", "roll_mean_28",
]

print("Panel rows:", f"{len(panel):,}")
print(panel[["lag_28", "same_dow_mean_4w", "roll_mean_28"]].describe().round(1))

Panel rows: 880,770
         lag_28  same_dow_mean_4w  roll_mean_28
count  854589.0          854589.0      842434.0
mean     5820.1            5803.7        5809.9
std      3905.5            3473.5        2180.1
min         0.0               0.0           0.0
25%      3712.0            4083.2        4377.6
50%      5774.0            5876.8        5478.3
75%      7926.0            7679.2        6745.1
max     38722.0           34594.2       24837.2


## 6. Baselines on the latest holdout

The latest holdout is the last 28 calendar days. Training ends the day before. Each model is scored on open days with positive sales.

In [6]:
def mape(actual, pred):
    return np.mean(np.abs(actual - pred) / actual) * 100

def wape(actual, pred):
    return np.sum(np.abs(actual - pred)) / np.sum(actual) * 100

def store_series(s):
    d = panel[panel["Store"] == s].set_index("Date")
    return d["Sales"], d["Open"]

def fill_for_fit(y):
    # Closed days already hold 0. Only missing dates are interpolated.
    return y.interpolate(limit_direction="both")

def same_weekday_forecast(sales, cut, index):
    hist = sales.loc[:cut].tail(HORIZON)
    by_day = hist.groupby(hist.index.dayofweek).mean()
    return by_day.reindex(index.dayofweek).values

def arima_forecast(y_train):
    y = np.log1p(fill_for_fit(y_train))
    model = SARIMAX(
        y, order=(1, 0, 1), seasonal_order=(1, 0, 1, 7),
        enforce_stationarity=False, enforce_invertibility=False,
    ).fit(disp=False)
    return np.expm1(model.forecast(HORIZON)).clip(lower=0).values

def prophet_forecast(y_train, index):
    d = fill_for_fit(y_train).reset_index()
    d.columns = ["ds", "y"]
    m = Prophet(weekly_seasonality=True, yearly_seasonality=True, daily_seasonality=False)
    m.fit(d)
    return m.predict(pd.DataFrame({"ds": index}))["yhat"].clip(lower=0).values

LAST_CUT = calendar[-HORIZON - 1]
HOLD_START = LAST_CUT + pd.Timedelta(days=1)
print("Latest cutoff:", LAST_CUT.date(), "| holdout starts:", HOLD_START.date())

rows = []
for s in TEST_STORES:
    sales, opened = store_series(s)
    y_tr = sales.loc[:LAST_CUT]
    y_te = sales.loc[HOLD_START:]
    idx = y_te.index
    keep = (opened.loc[HOLD_START:].eq(1) & y_te.notna() & (y_te > 0)).values
    actual = y_te.values[keep]

    preds = {
        "Same-weekday mean (4 weeks)": same_weekday_forecast(sales, LAST_CUT, idx),
        "ARIMA with weekly seasonality": arima_forecast(y_tr),
    }
    if HAS_PROPHET:
        preds["Prophet"] = prophet_forecast(y_tr, idx)

    for name, p in preds.items():
        p = np.asarray(p)[keep]
        rows.append({"Store": s, "Model": name,
                     "MAPE_%": mape(actual, p), "WAPE_%": wape(actual, p)})
    print("Done store", s)

baseline_results = pd.DataFrame(rows)
print(baseline_results.round(2).to_string(index=False))
print("\nAverage across test stores:")
print(baseline_results.groupby("Model")[["MAPE_%", "WAPE_%"]].mean().round(2).sort_values("MAPE_%"))

BASELINE_OUT = PROC / "baseline_results.csv"
baseline_results.to_csv(BASELINE_OUT, index=False)
print("Saved:", BASELINE_OUT)

Latest cutoff: 2015-07-03 | holdout starts: 2015-07-04
Done store 98
Done store 480
Done store 486
Done store 735
Done store 865
 Store                         Model  MAPE_%  WAPE_%
    98   Same-weekday mean (4 weeks)   17.47   16.25
    98 ARIMA with weekly seasonality   26.85   29.21
   480   Same-weekday mean (4 weeks)   12.63   12.87
   480 ARIMA with weekly seasonality   22.73   25.45
   486   Same-weekday mean (4 weeks)   22.27   20.59
   486 ARIMA with weekly seasonality   28.41   31.47
   735   Same-weekday mean (4 weeks)   21.71   21.15
   735 ARIMA with weekly seasonality   27.37   31.83
   865   Same-weekday mean (4 weeks)   16.46   15.51
   865 ARIMA with weekly seasonality   28.98   29.66

Average across test stores:
                               MAPE_%  WAPE_%
Model                                        
Same-weekday mean (4 weeks)     18.11   17.27
ARIMA with weekly seasonality   26.87   29.52
Saved: C:\Users\reinh\Downloads\Projects\Rossmann\processed\baseline_result

## 7. Gradient boosting on three rolling windows

One model is trained across all eligible stores for each window, using only open days up to that window's cutoff. Early stopping is off so the result is repeatable. The window-size check compares the scored store-days with the open days in the file, so a short window is caught.

In [7]:
def fit_model(cut, features):
    is_train = (panel["Date"] <= cut) & (panel["Open"] == 1) & panel["Sales"].notna()
    model = HistGradientBoostingRegressor(
        max_iter=400, learning_rate=0.05, max_leaf_nodes=31,
        min_samples_leaf=100, early_stopping=False, random_state=42,
    )
    model.fit(panel.loc[is_train, features], np.log1p(panel.loc[is_train, "Sales"]))
    return model, int(is_train.sum())

def naive_forecast(cut, ho):
    parts = []
    for s, d in ho.groupby("Store"):
        hist = panel[(panel["Store"] == s) & (panel["Date"] <= cut)].tail(HORIZON)
        by_dow = hist.groupby("DayOfWeek")["Sales"].mean()
        parts.append(pd.Series(d["DayOfWeek"].map(by_dow).values, index=d.index))
    return pd.concat(parts)

def evaluate_window(cut, features):
    model, n_train = fit_model(cut, features)
    end = cut + pd.Timedelta(days=HORIZON)
    ho = panel[
        (panel["Date"] > cut) & (panel["Date"] <= end) & panel["Store"].isin(TEST_STORES)
    ].copy()
    ho["pred_gb"] = np.expm1(model.predict(ho[features])).clip(min=0)
    ho["pred_naive"] = naive_forecast(cut, ho)

    # Window-size check: scored store-days against open days in the raw file
    expected = train[
        (train["Date"] > cut) & (train["Date"] <= end) & train["Store"].isin(TEST_STORES)
    ].groupby("Store")["Open"].sum()

    ho = ho[(ho["Open"] == 1) & (ho["Sales"] > 0)].copy()
    ho["cutoff"] = cut.date()
    scored = ho.groupby("Store").size()
    check = pd.DataFrame({"open_days_in_file": expected, "scored": scored}).fillna(0).astype(int)
    return ho[["cutoff", "Store", "Date", "Sales", "pred_gb", "pred_naive"]], n_train, check

frames = []
for off in WINDOW_OFFSETS:
    cut = calendar[-HORIZON - 1 - off]
    ho, n_train, check = evaluate_window(cut, FEATURES)
    print(f"\nCutoff {cut.date()}: trained on {n_train:,} rows, scored {len(ho):,} store-days")
    print("Open days in file vs scored, by store:")
    print(check.to_string())
    frames.append(ho)

daily = pd.concat(frames, ignore_index=True)
DAILY_OUT = PROC / "holdout_daily_predictions.csv"
daily.to_csv(DAILY_OUT, index=False)
print("\nSaved:", DAILY_OUT)


Cutoff 2015-07-03: trained on 709,900 rows, scored 120 store-days
Open days in file vs scored, by store:
       open_days_in_file  scored
Store                           
98                    24      24
480                   24      24
486                   24      24
735                   24      24
865                   24      24

Cutoff 2015-06-05: trained on 687,335 rows, scored 120 store-days
Open days in file vs scored, by store:
       open_days_in_file  scored
Store                           
98                    24      24
480                   24      24
486                   24      24
735                   24      24
865                   24      24

Cutoff 2015-05-08: trained on 667,088 rows, scored 107 store-days
Open days in file vs scored, by store:
       open_days_in_file  scored
Store                           
98                    21      21
480                   22      22
486                   21      21
735                   22      22
865                   

## 8. Accuracy across windows and head-to-head on the latest window

The same-weekday check confirms that the window version matches the baseline version on the latest window. Any difference means the setup has drifted.

In [8]:
rows = []
for (cutoff, s), d in daily.groupby(["cutoff", "Store"]):
    for name, col in [("Gradient boosting", "pred_gb"), ("Same-weekday mean", "pred_naive")]:
        rows.append({"cutoff": cutoff, "Store": s, "Model": name,
                     "MAPE_%": mape(d["Sales"], d[col]), "WAPE_%": wape(d["Sales"], d[col])})
acc = pd.DataFrame(rows)

print("Average per window:")
print(acc.groupby(["cutoff", "Model"])[["MAPE_%", "WAPE_%"]].mean().round(2))
print("\nAverage across all windows:")
print(acc.groupby("Model")[["MAPE_%", "WAPE_%"]].mean().round(2))
print("\nSpread of window averages (std of MAPE):")
print(acc.groupby(["cutoff", "Model"])["MAPE_%"].mean().groupby("Model").std().round(2))

wide = acc.pivot_table(index=["cutoff", "Store"], columns="Model", values="MAPE_%")
print("\nShare of store-windows where gradient boosting beats the same-weekday mean:",
      round(100 * (wide["Gradient boosting"] < wide["Same-weekday mean"]).mean(), 1), "%")

LAST_KEY = LAST_CUT.date()
check_same = acc[(acc["cutoff"] == LAST_KEY) & (acc["Model"] == "Same-weekday mean")][["Store", "MAPE_%"]]
check_same = check_same.merge(
    baseline_results[baseline_results["Model"] == "Same-weekday mean (4 weeks)"][["Store", "MAPE_%"]],
    on="Store", suffixes=("_window", "_baseline"),
)
print("\nSame-weekday check on the latest window (should match):")
print(check_same.round(2).to_string(index=False))

gb_last = acc[(acc["cutoff"] == LAST_KEY) & (acc["Model"] == "Gradient boosting")].copy()
gb_last["Model"] = "Gradient boosting (lags + calendar + store)"
head = pd.concat(
    [baseline_results, gb_last[["Store", "Model", "MAPE_%", "WAPE_%"]]], ignore_index=True
)
print("\nLatest window head-to-head (average across test stores):")
print(head.groupby("Model")[["MAPE_%", "WAPE_%"]].mean().round(2).sort_values("MAPE_%"))

Average per window:
                              MAPE_%  WAPE_%
cutoff     Model                            
2015-05-08 Gradient boosting    8.92    9.19
           Same-weekday mean   18.11   18.01
2015-06-05 Gradient boosting   10.17    9.76
           Same-weekday mean   23.55   23.94
2015-07-03 Gradient boosting    8.43    8.16
           Same-weekday mean   18.11   17.27

Average across all windows:
                   MAPE_%  WAPE_%
Model                            
Gradient boosting    9.18    9.04
Same-weekday mean   19.92   19.74

Spread of window averages (std of MAPE):
Model
Gradient boosting    0.90
Same-weekday mean    3.14
Name: MAPE_%, dtype: float64

Share of store-windows where gradient boosting beats the same-weekday mean: 100.0 %

Same-weekday check on the latest window (should match):
 Store  MAPE_%_window  MAPE_%_baseline
    98          17.47            17.47
   480          12.63            12.63
   486          22.27            22.27
   735          21.71       

## 8b. ARIMA on all three windows

The latest-window ARIMA result in section 6 is not enough on its own. This section fits the same univariate ARIMA for each test store in each window, so ARIMA can be compared on the same three windows as gradient boosting.

In [9]:
arima_rows = []
for off in WINDOW_OFFSETS:
    cut = calendar[-HORIZON - 1 - off]
    end = cut + pd.Timedelta(days=HORIZON)
    for s in TEST_STORES:
        sales, opened = store_series(s)
        y_tr = sales.loc[:cut]
        y_te = sales.loc[cut + pd.Timedelta(days=1):end]
        keep = (opened.loc[y_te.index].eq(1) & y_te.notna() & (y_te > 0)).values
        pred = arima_forecast(y_tr)[keep]
        actual = y_te.values[keep]
        arima_rows.append({"cutoff": cut.date(), "Store": s, "Model": "ARIMA with weekly seasonality",
                           "MAPE_%": mape(actual, pred), "WAPE_%": wape(actual, pred)})
arima_all = pd.DataFrame(arima_rows)

print("ARIMA average per window:")
print(arima_all.groupby("cutoff")[["MAPE_%", "WAPE_%"]].mean().round(2))
print("\nARIMA average across windows:")
print(arima_all[["MAPE_%", "WAPE_%"]].mean().round(2))

ARIMA_OUT = PROC / "arima_all_windows.csv"
arima_all.round(2).to_csv(ARIMA_OUT, index=False)
print("Saved:", ARIMA_OUT)

ARIMA average per window:
            MAPE_%  WAPE_%
cutoff                    
2015-05-08   29.55   32.17
2015-06-05   30.72   34.18
2015-07-03   26.87   29.52

ARIMA average across windows:
MAPE_%    29.05
WAPE_%    31.96
dtype: float64
Saved: C:\Users\reinh\Downloads\Projects\Rossmann\processed\arima_all_windows.csv


## 9. Promotion ablation

This drops the promotion and holiday features and reruns all three windows. If accuracy stays close to the full model, the lags and store structure are doing most of the work. If it moves towards the same-weekday mean, promotions were the main source of the gain.

In [10]:
PROMO_FEATURES = ["Promo", "Promo2", "SchoolHoliday", "StateHoliday"]
NO_PROMO_FEATURES = [f for f in FEATURES if f not in PROMO_FEATURES]

ablation_rows = []
for off in WINDOW_OFFSETS:
    cut = calendar[-HORIZON - 1 - off]
    ho, _, _ = evaluate_window(cut, NO_PROMO_FEATURES)
    for s, d in ho.groupby("Store"):
        ablation_rows.append({"cutoff": cut.date(), "Store": s,
                              "Model": "Gradient boosting without promo/holiday features",
                              "MAPE_%": mape(d["Sales"], d["pred_gb"]),
                              "WAPE_%": wape(d["Sales"], d["pred_gb"])})
ablation = pd.DataFrame(ablation_rows)

full = acc[acc["Model"] == "Gradient boosting"].copy()
full["Model"] = "Gradient boosting (full)"
compare_ablation = pd.concat([full, ablation], ignore_index=True)
print(compare_ablation.groupby(["cutoff", "Model"])[["MAPE_%", "WAPE_%"]].mean().round(2))
print("\nAverage across windows:")
print(compare_ablation.groupby("Model")[["MAPE_%", "WAPE_%"]].mean().round(2))

ABLATION_OUT = PROC / "ablation_results.csv"
compare_ablation.round(2).to_csv(ABLATION_OUT, index=False)
print("Saved:", ABLATION_OUT)

                                                             MAPE_%  WAPE_%
cutoff     Model                                                           
2015-05-08 Gradient boosting (full)                            8.92    9.19
           Gradient boosting without promo/holiday features    9.72    9.94
2015-06-05 Gradient boosting (full)                           10.17    9.76
           Gradient boosting without promo/holiday features   12.72   12.76
2015-07-03 Gradient boosting (full)                            8.43    8.16
           Gradient boosting without promo/holiday features    9.65    9.19

Average across windows:
                                                  MAPE_%  WAPE_%
Model                                                           
Gradient boosting (full)                            9.18    9.04
Gradient boosting without promo/holiday features   10.70   10.63
Saved: C:\Users\reinh\Downloads\Projects\Rossmann\processed\ablation_results.csv


## 10. Inventory cost

The data has no unit costs, so the cost uses stated assumptions, in euros of sales value.

- **Too much stock (over-forecast):** unsold stock is written off. The loss is the purchase cost, (1 - margin) of sales value.
- **Too little stock (under-forecast):** the sale is lost. The loss is the margin on that sale.

The margin is tested at 15%, 25% and 35% because the real figure is unknown. A positive saving favours gradient boosting.

Limitations: one day's stock is the unit of decision, holding over several days is ignored, and the results cover five stores across three windows. Scaling to the chain needs a separate check.

In [11]:
def stock_cost(d, pred_col, margin):
    over = (d[pred_col] - d["Sales"]).clip(lower=0)
    under = (d["Sales"] - d[pred_col]).clip(lower=0)
    return float(((1 - margin) * over + margin * under).sum())

total_sales = daily["Sales"].sum()
cost_rows = []
for m in MARGINS:
    for name, col in [("Gradient boosting", "pred_gb"), ("Same-weekday mean", "pred_naive")]:
        c = stock_cost(daily, col, m)
        cost_rows.append({
            "margin": m, "Model": name, "cost_eur": round(c),
            "cost_pct_of_sales": round(100 * c / total_sales, 2),
            "cost_eur_per_store_day": round(c / len(daily), 2),
        })
cost = pd.DataFrame(cost_rows)
print(cost.to_string(index=False))

pivot = cost.pivot(index="margin", columns="Model", values="cost_eur")
pivot["saving_eur"] = pivot["Same-weekday mean"] - pivot["Gradient boosting"]
pivot["saving_pct"] = (100 * pivot["saving_eur"] / pivot["Same-weekday mean"]).round(1)
print("\nSaving from gradient boosting compared with the same-weekday mean:")
print(pivot.round(0).to_string())

COST_OUT = PROC / "inventory_cost_results.csv"
cost.to_csv(COST_OUT, index=False)
print("\nSaved:", COST_OUT)

 margin             Model  cost_eur  cost_pct_of_sales  cost_eur_per_store_day
   0.15 Gradient boosting     90481               4.54                  260.75
   0.15 Same-weekday mean    179866               9.03                  518.35
   0.25 Gradient boosting     90725               4.56                  261.46
   0.25 Same-weekday mean    184414               9.26                  531.45
   0.35 Gradient boosting     90970               4.57                  262.16
   0.35 Same-weekday mean    188962               9.49                  544.56

Saving from gradient boosting compared with the same-weekday mean:
Model   Gradient boosting  Same-weekday mean  saving_eur  saving_pct
margin                                                              
0.15                90481             179866       89385        50.0
0.25                90725             184414       93689        51.0
0.35                90970             188962       97992        52.0

Saved: C:\Users\reinh\Downloads\P